# CrackFusion-Tiny — checkpointed Colab training

Select a **T4 GPU**, then **Run all** and authorize Drive once. This notebook contains the complete model and training code; no repository, `src` files, or project requirements file are needed. Standard Colab PyTorch, torchvision, NumPy, Pillow and scikit-learn packages are used.

Notebook 01 exported a balanced **3,600-image development split**: 2,880 training and 720 validation images. The setup reads `Fridge/major_project_dataset/crackfusion_colab_bundle.zip` automatically (also supports the audit output folder or a runtime upload), copies images to Colab's local disk, and verifies their audit hashes before training.

Checkpoints, normalization progress, configuration, predictions and metrics are saved to **MyDrive/Fridge/major_project_dataset/outputs/crackfusion_tiny**. Rerun with the same configuration to resume from the last completed epoch; an interrupted epoch restarts. Keep the same PyTorch/runtime configuration when resuming. Three seeds train for up to 100 epochs with early stopping.

The audit has no verified parent-photo mapping. Results retain `tile_development_only`; they are development validation results. Synthetic software checks are optional (`RUN_SOFTWARE_CHECKS=False` by default).

## 2. Environment


In [ ]:
import os, csv, hashlib, io, json, math, random, statistics, shutil, sys, time
from pathlib import Path
from tempfile import TemporaryDirectory
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
import torchvision
from zipfile import ZipFile
from torchvision import transforms as T
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score, average_precision_score)

DRIVE_DATA_DIR = None
try:
    from google.colab import drive
except ImportError:
    pass
else:
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    DRIVE_DATA_DIR = Path("/content/drive/MyDrive/Fridge/major_project_dataset")
    if not DRIVE_DATA_DIR.is_dir():
        raise FileNotFoundError("Mount the Google account containing MyDrive/Fridge/major_project_dataset.")

RUN_SOFTWARE_CHECKS = False  # Enable only to run the optional synthetic checks.
validation_report = {}

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else
                      'mps' if torch.backends.mps.is_available() else 'cpu')
torch.set_num_threads(min(4, torch.get_num_threads()))
seed_everything(42)
print({'torch': str(torch.__version__), 'torchvision': str(torchvision.__version__),
       'device': str(DEVICE), 'cpu_threads': torch.get_num_threads()})


## 3. Complete model

All shapes use PyTorch's batch/channel/height/width order. The two branches receive
the **same** stem output. Tokens return to their original spatial ordering before
fusion. `return_features=True` exposes branch tensors for inspection or attribution.
The model returns a raw logit; sigmoid belongs in inference, not before BCE loss.
Input resolution is deliberately fixed and checked rather than silently reshaped.


In [ ]:
"""CrackFusion-Tiny model architecture."""

import torch
from torch import nn
import torch.nn.functional as F


class ConvBNGELU(nn.Sequential):
    def __init__(self, cin: int, cout: int, kernel: int = 3, stride: int = 1):
        super().__init__(
            nn.Conv2d(cin, cout, kernel, stride, kernel // 2, bias=False),
            nn.BatchNorm2d(cout),
            nn.GELU(),
        )


class DSResidual(nn.Module):
    def __init__(self, cin: int, cout: int, stride: int = 1):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(cin, cin, 3, stride, 1, groups=cin, bias=False),
            nn.BatchNorm2d(cin),
            nn.GELU(),
            nn.Conv2d(cin, cout, 1, bias=False),
            nn.BatchNorm2d(cout),
        )
        self.skip = (
            nn.Identity()
            if cin == cout and stride == 1
            else nn.Sequential(
                nn.Conv2d(cin, cout, 1, stride, bias=False),
                nn.BatchNorm2d(cout),
            )
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.gelu(self.body(x) + self.skip(x))


class CrackFusionTiny(nn.Module):
    """Parallel local/global classifier for 224×224 RGB images."""

    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            ConvBNGELU(3, 32, stride=2),
            DSResidual(32, 48, 2),
            DSResidual(48, 64, 2),
        )
        self.local = nn.Sequential(
            DSResidual(64, 96),
            DSResidual(96, 96),
            nn.AvgPool2d(2),
        )
        self.tokenizer = ConvBNGELU(64, 96, stride=2)
        self.position = nn.Parameter(torch.empty(1, 96, 14, 14))
        nn.init.trunc_normal_(self.position, std=0.02)
        self.encoders = nn.ModuleList(
            [
                nn.TransformerEncoderLayer(
                    96,
                    4,
                    dim_feedforward=192,
                    dropout=0.1,
                    activation="gelu",
                    batch_first=True,
                    norm_first=True,
                )
                for _ in range(2)
            ]
        )
        self.token_norm = nn.LayerNorm(96)
        self.fusion = ConvBNGELU(192, 128, kernel=1)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
        )

    def forward(self, x: torch.Tensor, return_features: bool = False):
        if x.ndim != 4 or tuple(x.shape[1:]) != (3, 224, 224) or x.shape[0] == 0:
            raise ValueError("Expected nonempty floating-point input [B, 3, 224, 224]")
        if not x.is_floating_point():
            raise TypeError("Preprocess pixels into a floating-point tensor first")

        shared = self.stem(x)
        local = self.local(shared)
        token_map = self.tokenizer(shared)
        tokens = (token_map + self.position).flatten(2).transpose(1, 2)
        for encoder in self.encoders:
            tokens = encoder(tokens)
        tokens = self.token_norm(tokens)
        global_map = tokens.transpose(1, 2).reshape(x.shape[0], 96, 14, 14)
        fused = self.fusion(torch.cat((local, global_map), dim=1))
        logits = self.head(fused)

        if return_features:
            return logits, {
                "stem": shared,
                "local": local,
                "tokens": tokens,
                "global_map": global_map,
                "fused": fused,
            }
        return logits


def count_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)



model = CrackFusionTiny()
PARAMETERS = count_parameters(model)
print(f'Trainable parameters: {PARAMETERS:,} ({PARAMETERS / 1e6:.3f}M)')
print(f'Parameter storage in float32: {PARAMETERS * 4 / 1024**2:.2f} MiB')
with torch.no_grad():
    logits, features = model.eval()(torch.randn(2, 3, 224, 224), return_features=True)
print({name: list(value.shape) for name, value in features.items()})
print('logits:', list(logits.shape))

assert PARAMETERS == 284337


## 4. Optional architecture checks

Enable `RUN_SOFTWARE_CHECKS` in the setup cell to execute these checks.


In [ ]:
def validate_model():
    seed_everything(42)
    net = CrackFusionTiny()
    x = torch.randn(2, 3, 224, 224)
    y = torch.tensor([[0.], [1.]])
    scores, maps = net(x, return_features=True)
    expected = {'stem': (2, 64, 28, 28), 'local': (2, 96, 14, 14),
                'tokens': (2, 196, 96), 'global_map': (2, 96, 14, 14),
                'fused': (2, 128, 14, 14)}
    assert scores.shape == (2, 1)
    assert all(tuple(maps[k].shape) == v for k, v in expected.items())
    loss = nn.BCEWithLogitsLoss()(scores, y)
    loss.backward()
    assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in net.parameters())
    for module in (net.stem, net.local, net.tokenizer, net.encoders, net.fusion, net.head):
        assert sum(p.grad.abs().sum().item() for p in module.parameters()) > 0
    assert net.position.grad.abs().sum() > 0
    assert not torch.equal(net.encoders[0].self_attn.in_proj_weight,
                           net.encoders[1].self_attn.in_proj_weight)
    net.eval()
    with torch.no_grad():
        reference = net(x)
        torch.testing.assert_close(net(x[:1]), reference[:1], rtol=1e-4, atol=1e-5)
        torch.testing.assert_close(net(x), reference)
    buffer = io.BytesIO()
    torch.save(net.state_dict(), buffer)
    buffer.seek(0)
    restored = CrackFusionTiny().eval()
    restored.load_state_dict(torch.load(buffer, weights_only=True))
    with torch.no_grad():
        torch.testing.assert_close(restored(x), reference)
    for bad in (torch.randn(2, 3, 256, 256), torch.randn(2, 1, 224, 224),
                torch.empty(0, 3, 224, 224)):
        try:
            net(bad)
        except ValueError:
            pass
        else:
            raise AssertionError('Bad shape accepted')
    device_net = CrackFusionTiny().to(DEVICE).train()
    device_loss = nn.BCEWithLogitsLoss()(device_net(x.to(DEVICE)), y.to(DEVICE))
    device_loss.backward()
    assert torch.isfinite(device_loss)
    assert all(p.grad is not None and torch.isfinite(p.grad).all()
               for p in device_net.parameters())
    return {'shape_checks': 'passed', 'all_parameter_gradients': 'passed',
            'checkpoint_round_trip': 'passed', 'batch_independence': 'passed',
            'input_validation': 'passed', 'device_backward': str(DEVICE)}

if RUN_SOFTWARE_CHECKS:
    validation_report = validate_model()
    print(json.dumps(validation_report, indent=2))


## 5. Optional synthetic learning check

This artificial stripe task checks optimization only.


In [ ]:
if RUN_SOFTWARE_CHECKS:
    seed_everything(42)
    toy_x = (0.6 + 0.08 * torch.randn(8, 3, 224, 224)).clamp(0, 1)
    toy_y = torch.tensor([0, 1] * 4, dtype=torch.float32).view(-1, 1)
    for i in range(1, 8, 2):
        toy_x[i, :, :, 105:113] = 0.05
    toy_model = CrackFusionTiny().train()
    toy_optimizer = torch.optim.AdamW(toy_model.parameters(), lr=1e-3)
    toy_loss_fn = nn.BCEWithLogitsLoss()
    toy_losses = []
    for step in range(50):
        toy_optimizer.zero_grad(set_to_none=True)
        toy_loss = toy_loss_fn(toy_model(toy_x), toy_y)
        assert torch.isfinite(toy_loss)
        toy_loss.backward()
        nn.utils.clip_grad_norm_(toy_model.parameters(), 1.0)
        toy_optimizer.step()
        toy_losses.append(toy_loss.item())
    toy_model.eval()
    with torch.no_grad():
        toy_logits = toy_model(toy_x)
        toy_accuracy = ((toy_logits >= 0) == toy_y.bool()).float().mean().item()
    assert np.mean(toy_losses[-5:]) < 0.5 * np.mean(toy_losses[:5])
    assert toy_accuracy >= 0.875
    validation_report['toy_first_loss'] = toy_losses[0]
    validation_report['toy_last_loss'] = toy_losses[-1]
    validation_report['toy_training_accuracy'] = toy_accuracy
    print({k: v for k, v in validation_report.items() if k.startswith('toy')})
    plt.plot(toy_losses)
    plt.xlabel('Optimization step'); plt.ylabel('BCE loss')
    plt.title('Artificial stripe memorization — software check only'); plt.show()
    del toy_model, toy_optimizer


## 6. Audit data and training configuration

The completed audit manifest supplies labels, hashes and development splits. Images are cached on local disk; results go to Drive. Normalization uses training images only. Training starts in section 7.


In [ ]:
def prepare_colab_data(base):
    audit_dir = base / "outputs" / "data_audit"
    candidates = [base / "crackfusion_colab_bundle.zip",
                  audit_dir / "crackfusion_colab_bundle.zip",
                  Path("/content/crackfusion_colab_bundle.zip")]
    source = next((p for p in candidates if p.is_file()), None)
    cache = Path("/content/crackfusion_data")
    cache.mkdir(parents=True, exist_ok=True)
    manifest = cache / "development_manifest.csv"
    data_root = cache / "dataset" / "SDNET2018"
    if source is not None:
        local_zip = cache / "bundle.zip"
        # Copy the small bundle once; image reads during training stay on local disk.
        if source != local_zip:
            temporary = local_zip.with_suffix(".zip.tmp")
            shutil.copyfile(source, temporary)
            if temporary.stat().st_size != source.stat().st_size:
                raise IOError("Incomplete bundle copy; rerun this cell.")
            temporary.replace(local_zip)
        digest = hashlib.sha256(local_zip.read_bytes()).hexdigest()
        marker = cache / ".extracted_sha256"
        if not marker.is_file() or marker.read_text() != digest:
            with ZipFile(local_zip) as bundle:
                root = cache.resolve()
                for member in bundle.infolist():
                    if not (cache / member.filename).resolve().is_relative_to(root):
                        raise ValueError(f"Unsafe bundle path: {member.filename}")
                bundle.extractall(cache)
            marker.write_text(digest)
    else:
        original_manifest = audit_dir / "development_manifest.csv"
        if not original_manifest.is_file():
            raise FileNotFoundError(
                f"Audit export is missing in {base}. Save crackfusion_colab_bundle.zip "
                "there from notebook 01, or upload that bundle to /content. "
                "Saved notebook output alone does not guarantee that Drive finished syncing.")
        shutil.copyfile(original_manifest, manifest)
        # Audit paths may point at a different computer; find the actual Drive dataset.
        roots = [base / "SDNET2018" / "SDNET2018", base / "SDNET2018", base / "dataset" / "SDNET2018"]
        source_root = next((r for r in roots if (r / "D" / "CD").is_dir()), None)
        if source_root is None:
            raise FileNotFoundError("No extracted SDNET2018 dataset or training bundle in the Drive folder.")
        with manifest.open(newline="") as stream:
            rows = list(csv.DictReader(stream))
        for index, row in enumerate(rows):
            relative = Path(row["path"])
            source_image, target = (source_root / relative).resolve(), (data_root / relative).resolve()
            if not source_image.is_relative_to(source_root.resolve()) or not target.is_relative_to(data_root.resolve()):
                raise ValueError("Unsafe manifest path")
            expected = row.get("sha256", "")
            if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest() != expected:
                target.parent.mkdir(parents=True, exist_ok=True)
                temporary = target.with_suffix(target.suffix + ".tmp")
                shutil.copyfile(source_image, temporary)
                temporary.replace(target)
            if (index + 1) % 500 == 0:
                print(f"Cached {index + 1}/{len(rows)} images")
    if not manifest.is_file() or not data_root.is_dir():
        raise FileNotFoundError("Bundle must contain development_manifest.csv and dataset/SDNET2018 images.")
    with manifest.open(newline="") as stream:
        rows = list(csv.DictReader(stream))
    counts = {split: sum(r["split"] == split for r in rows) for split in ("dev_train", "dev_validation")}
    if counts != {"dev_train": 2880, "dev_validation": 720} or len(rows) != 3600:
        raise ValueError(f"Unexpected audit development split: {counts}; expected 2880 train / 720 validation.")
    if any(r.get("split_basis") != "tile_development_only" or not r.get("sha256") for r in rows):
        raise ValueError("Use the hashed development manifest exported by notebook 01.")
    summary_path = audit_dir / "data_audit_summary.json"
    if summary_path.is_file():
        summary = json.loads(summary_path.read_text())
        if summary.get("development_images") != len(rows):
            raise ValueError("Audit summary and bundle disagree on development image count.")
        original = audit_dir / "development_manifest.csv"
        if original.is_file() and original.read_bytes() != manifest.read_bytes():
            raise ValueError("Drive audit manifest and bundle differ. Re-export the bundle from notebook 01.")
    print("Audit split:", counts, "(tile_development_only)")
    print("Local images:", data_root)
    return data_root, manifest

if DRIVE_DATA_DIR:
    DATA_ROOT, MANIFEST = prepare_colab_data(DRIVE_DATA_DIR)
    OUTPUT_ROOT = DRIVE_DATA_DIR / "outputs" / "crackfusion_tiny"
    PROTOCOL = "development"
    RUN_TRAINING = True
else:
    DATA_ROOT = Path("dataset/SDNET2018")
    MANIFEST = Path("outputs/data_audit/development_manifest.csv")
    OUTPUT_ROOT = Path("outputs/crackfusion_tiny")
    PROTOCOL = "development"
    RUN_TRAINING = False
PARENT_ID_PROVENANCE = ""  # describe the verified mapping's source
RUN_TEST = False
SEEDS = (42, 1337, 2026)
BATCH_SIZE = 16
EFFECTIVE_BATCH = 64
MAX_EPOCHS = 100
WARMUP_EPOCHS = 5
MIN_EPOCHS = 30
PATIENCE = 15
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-2
THRESHOLD = 0.5


In [ ]:
def save_json(path, value):
    temporary = path.with_name(path.name + '.tmp')
    with temporary.open('w', encoding='utf-8') as stream:
        json.dump(value, stream, indent=2)
        stream.flush()
        os.fsync(stream.fileno())
    temporary.replace(path)

def save_checkpoint(path, value):
    temporary = path.with_name(path.name + '.tmp')
    with temporary.open('wb') as stream:
        torch.save(value, stream)
        stream.flush()
        os.fsync(stream.fileno())
    temporary.replace(path)

def load_manifest(path, root, protocol, provenance=''):
    if protocol not in {'source_separated', 'development'}:
        raise ValueError('Unknown protocol')
    with Path(path).open(newline='') as stream:
        reader = csv.DictReader(stream)
        if not {'path', 'label', 'surface', 'parent_id', 'split'} <= set(reader.fieldnames or []):
            raise ValueError('Manifest needs path,label,surface,parent_id,split')
        rows = list(reader)
    if not rows:
        raise ValueError('Empty manifest')
    root = Path(root).resolve()
    seen_paths, hash_splits, parent_splits = set(), {}, {}
    for row in rows:
        split_aliases = {'val': 'validation'}
        if protocol == 'development':
            split_aliases.update(dev_train='train', dev_validation='validation')
        row['split'] = split_aliases.get(row['split'], row['split'])
        if row['split'] not in {'train', 'validation', 'test'} or row['label'] not in {'0', '1'}:
            raise ValueError('Invalid split or binary label')
        if row['surface'] not in {'deck', 'wall', 'pavement'}:
            raise ValueError('Unknown surface')
        image_path = (root / row['path']).resolve()
        if not image_path.is_relative_to(root) or not image_path.is_file():
            raise ValueError(f'Image missing or outside DATA_ROOT: {image_path}')
        if image_path in seen_paths:
            raise ValueError('Repeated image path')
        seen_paths.add(image_path)
        digest = hashlib.sha256(image_path.read_bytes()).hexdigest()
        if row.get('sha256') and digest != row['sha256']:
            raise ValueError(f"Image differs from audit hash: {row['path']}")
        if row.get('excluded', '').lower() in {'true', '1'}:
            raise ValueError('Excluded audit image in training manifest')
        previous = hash_splits.setdefault(digest, row['split'])
        if previous != row['split']:
            raise ValueError('Exact duplicate image crosses splits')
        with Image.open(image_path) as image:
            image.verify()
        parent = row['parent_id'].strip()
        if protocol == 'source_separated':
            if not parent or not provenance.strip():
                raise ValueError('Verified parent IDs and provenance are required')
            if parent_splits.setdefault(parent, row['split']) != row['split']:
                raise ValueError(f'Parent leakage: {parent}')
        row['label'] = int(row['label'])
    groups = {s: [r for r in rows if r['split'] == s] for s in ('train', 'validation', 'test')}
    required = ('train', 'validation', 'test') if protocol == 'source_separated' else ('train', 'validation')
    for split in required:
        if {r['label'] for r in groups[split]} != {0, 1}:
            raise ValueError(f'{split} must contain both classes')
    return groups

class CrackDataset(Dataset):
    def __init__(self, rows, root, transform):
        self.rows, self.root, self.transform = rows, Path(root), transform
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        with Image.open(self.root / row['path']) as image:
            tensor = self.transform(image.convert('RGB'))
        return tensor, torch.tensor([row['label']], dtype=torch.float32)

def training_normalization(rows, root, checkpoint_path=None):
    transform = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor()])
    loader = DataLoader(CrackDataset(rows, root, transform), batch_size=BATCH_SIZE)
    sums, squares, pixels = torch.zeros(3, dtype=torch.float64), torch.zeros(3, dtype=torch.float64), 0
    start_batch = 0
    if checkpoint_path is not None and checkpoint_path.exists():
        saved = json.loads(checkpoint_path.read_text())
        sums = torch.tensor(saved['sums'], dtype=torch.float64)
        squares = torch.tensor(saved['squares'], dtype=torch.float64)
        pixels, start_batch = saved['pixels'], saved['next_batch']
    for batch_index, (images, _) in enumerate(loader):
        if batch_index < start_batch:
            continue
        images = images.double()
        sums += images.sum((0, 2, 3))
        squares += images.square().sum((0, 2, 3))
        pixels += images.shape[0] * images.shape[2] * images.shape[3]
        if checkpoint_path is not None:
            save_json(checkpoint_path, dict(sums=sums.tolist(), squares=squares.tolist(),
                                          pixels=pixels, next_batch=batch_index + 1))
    mean = sums / pixels
    std = (squares / pixels - mean.square()).clamp_min(1e-8).sqrt()
    return mean.tolist(), std.tolist()

def make_transforms(mean, std):
    train = T.Compose([T.RandomResizedCrop(224, scale=(0.85, 1.0)),
        T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
        T.RandomRotation(15, interpolation=T.InterpolationMode.BILINEAR, fill=tuple(round(m*255) for m in mean)),
        T.ColorJitter(brightness=0.15, contrast=0.15), T.ToTensor(), T.Normalize(mean, std)])
    evaluation = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(mean, std)])
    return train, evaluation

def binary_metrics(labels, probabilities, threshold=0.5):
    labels, probabilities = np.asarray(labels).reshape(-1), np.asarray(probabilities).reshape(-1)
    predictions = probabilities >= threshold
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    both = len(np.unique(labels)) == 2
    return dict(accuracy=float(accuracy_score(labels, predictions)),
        balanced_accuracy=float(balanced_accuracy_score(labels, predictions)),
        precision=float(precision_score(labels, predictions, zero_division=0)),
        recall=float(recall_score(labels, predictions, zero_division=0)),
        specificity=float(tn / (tn + fp)) if tn + fp else None,
        f1=float(f1_score(labels, predictions, zero_division=0)),
        macro_f1=float(f1_score(labels, predictions, labels=[0, 1], average='macro', zero_division=0)),
        roc_auc=float(roc_auc_score(labels, probabilities)) if both else None,
        average_precision=float(average_precision_score(labels, probabilities)) if both else None,
        confusion_matrix=[[int(tn), int(fp)], [int(fn), int(tp)]])

@torch.inference_mode()
def evaluate(net, loader, criterion):
    net.eval()
    labels, probabilities, total = [], [], 0.
    for images, targets in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        logits = net(images)
        total += criterion(logits, targets).item() * len(images)
        labels.extend(targets.cpu().flatten().tolist())
        probabilities.extend(logits.sigmoid().cpu().flatten().tolist())
    return total / len(loader.dataset), binary_metrics(labels, probabilities), probabilities


## 7. Training, checkpointing and validation

Selection uses validation macro-F1. Weighted BCE uses training class counts; CUDA uses AMP. Each configuration has a stable output folder, with best inference weights and a last-epoch checkpoint containing optimizer, scaler and random states. Every completed epoch is saved to Drive. Rerun the same configuration to resume.


In [ ]:
def step_optimizer(net, optimizer, scaler):
    scaler.unscale_(optimizer)
    gradients = [p.grad for p in net.parameters() if p.grad is not None]
    finite = bool(torch.stack([torch.isfinite(g).all() for g in gradients]).all().item())
    if finite:
        nn.utils.clip_grad_norm_(net.parameters(), 1.0, error_if_nonfinite=True)
    elif not scaler.is_enabled():
        raise FloatingPointError('Nonfinite gradients with AMP disabled')
    # unscale_ recorded any overflow: GradScaler skips the update and lowers its scale.
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    return finite

def train_one_seed(groups, mean, std, seed, run_dir):
    seed_everything(seed)
    train_tf, eval_tf = make_transforms(mean, std)
    loaders = {split: DataLoader(CrackDataset(rows, DATA_ROOT,
                    train_tf if split == 'train' else eval_tf), batch_size=BATCH_SIZE,
                    shuffle=split == 'train', num_workers=0,
                    generator=torch.Generator().manual_seed(seed))
               for split, rows in groups.items() if rows}
    net = CrackFusionTiny().to(DEVICE)
    positive = sum(r['label'] for r in groups['train'])
    weight = (len(groups['train']) - positive) / positive
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([weight], device=DEVICE))
    optimizer = torch.optim.AdamW(net.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')
    accumulation = EFFECTIVE_BATCH // BATCH_SIZE
    best, stale, history = -1., 0, []
    checkpoint_path = run_dir / f'best_seed_{seed}.pt'
    resume_path = run_dir / f'last_seed_{seed}.pt'
    start_epoch = 0
    if resume_path.exists():
        saved = torch.load(resume_path, map_location='cpu', weights_only=True)
        net.load_state_dict(saved['model'])
        optimizer.load_state_dict(saved['optimizer'])
        scaler.load_state_dict(saved['scaler'])
        start_epoch, best, stale, history = saved['next_epoch'], saved['best'], saved['stale'], saved['history']
        random.setstate(saved['python_rng'])
        numpy_rng = saved['numpy_rng']
        np.random.set_state((numpy_rng[0], np.asarray(numpy_rng[1], dtype=np.uint32), *numpy_rng[2:]))
        torch.set_rng_state(saved['torch_rng'])
        if DEVICE.type == 'cuda' and saved['cuda_rng']:
            torch.cuda.set_rng_state_all(saved['cuda_rng'])
        for split, state in saved['loader_rng'].items():
            loaders[split].generator.set_state(state)
        print(f'Resuming seed={seed} after epoch={start_epoch}')
    for epoch in range(start_epoch, MAX_EPOCHS):
        if epoch >= MIN_EPOCHS and stale >= PATIENCE:
            break
        factor = ((epoch + 1) / WARMUP_EPOCHS if epoch < WARMUP_EPOCHS else
                  0.5 * (1 + math.cos(math.pi * (epoch - WARMUP_EPOCHS) /
                                     max(1, MAX_EPOCHS - WARMUP_EPOCHS - 1))))
        for group in optimizer.param_groups:
            group['lr'] = LEARNING_RATE * factor
        net.train()
        optimizer.zero_grad(set_to_none=True)
        total = 0.
        amp_skipped = 0
        for index, (images, targets) in enumerate(loaders['train']):
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            start = (index // accumulation) * accumulation * BATCH_SIZE
            group_size = min(EFFECTIVE_BATCH, len(groups['train']) - start)
            with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == 'cuda'):
                loss = criterion(net(images), targets)
            if not torch.isfinite(loss):
                raise FloatingPointError('Nonfinite training loss')
            scaler.scale(loss * len(images) / group_size).backward()
            if (index + 1) % accumulation == 0 or index + 1 == len(loaders['train']):
                if not step_optimizer(net, optimizer, scaler):
                    amp_skipped += 1
                    print(f'AMP overflow: skipped update; scale reduced to {scaler.get_scale():g}')
            total += loss.item() * len(images)
        val_loss, metrics, _ = evaluate(net, loaders['validation'], criterion)
        record = dict(epoch=epoch+1, amp_skipped_updates=amp_skipped, train_loss=total/len(groups['train']),
                      validation_loss=val_loss, lr=optimizer.param_groups[0]['lr'], **metrics)
        history.append(record)
        print(f"seed={seed} epoch={epoch+1} loss={record['train_loss']:.4f} val_macro_f1={metrics['macro_f1']:.4f}")
        if metrics['macro_f1'] > best:
            best, stale = metrics['macro_f1'], 0
            save_checkpoint(checkpoint_path, dict(model=net.state_dict(), seed=seed, epoch=epoch+1,
                architecture='CrackFusionTiny-v1', parameters=PARAMETERS,
                mean=mean, std=std, threshold=THRESHOLD, protocol=PROTOCOL,
                validation_metrics=metrics))
        else:
            stale += 1
        numpy_rng = np.random.get_state()
        save_checkpoint(resume_path, dict(model=net.state_dict(), optimizer=optimizer.state_dict(),
            scaler=scaler.state_dict(), next_epoch=epoch + 1, best=best, stale=stale, history=history,
            python_rng=random.getstate(),
            numpy_rng=(numpy_rng[0], numpy_rng[1].tolist(), *numpy_rng[2:]),
            torch_rng=torch.get_rng_state(),
            cuda_rng=torch.cuda.get_rng_state_all() if DEVICE.type == 'cuda' else [],
            loader_rng={split: loader.generator.get_state() for split, loader in loaders.items()}))
        save_json(run_dir / f'history_seed_{seed}.json', history)
        if epoch + 1 >= MIN_EPOCHS and stale >= PATIENCE:
            break
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
    net.load_state_dict(checkpoint['model'])
    split = 'test' if RUN_TEST else 'validation'
    _, metrics, probabilities = evaluate(net, loaders[split], criterion)
    per_surface = {}
    for surface in ('deck', 'wall', 'pavement'):
        selected = [(r['label'], p) for r, p in zip(groups[split], probabilities) if r['surface'] == surface]
        if selected:
            ys, ps = zip(*selected)
            per_surface[surface] = binary_metrics(ys, ps)
    result = dict(seed=seed, split=split, protocol=PROTOCOL, metrics=metrics, per_surface=per_surface)

    prediction_path = run_dir / f'predictions_seed_{seed}.csv'
    temporary_predictions = prediction_path.with_suffix('.csv.tmp')
    with temporary_predictions.open('w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=['path', 'label', 'surface', 'parent_id', 'probability', 'prediction'])
        writer.writeheader()
        for row, probability in zip(groups[split], probabilities):
            writer.writerow({k: row[k] for k in ('path', 'label', 'surface', 'parent_id')} |
                            dict(probability=probability, prediction=int(probability >= THRESHOLD)))
    temporary_predictions.replace(prediction_path)
    # Mark a seed complete only after its predictions have been written.
    save_json(run_dir / f'metrics_seed_{seed}.json', result)
    return result

def run_experiment():
    if BATCH_SIZE <= 0 or EFFECTIVE_BATCH % BATCH_SIZE or MAX_EPOCHS < 1:
        raise ValueError('Batch size must divide effective batch; epochs must be positive')
    if RUN_TEST and PROTOCOL != 'source_separated':
        raise ValueError('Development splits cannot produce research test results')
    manifest_bytes = MANIFEST.read_bytes()
    configuration = {key: globals()[key] for key in ('PROTOCOL', 'PARENT_ID_PROVENANCE', 'SEEDS',
        'BATCH_SIZE', 'EFFECTIVE_BATCH', 'MAX_EPOCHS', 'WARMUP_EPOCHS', 'MIN_EPOCHS',
        'PATIENCE', 'LEARNING_RATE', 'WEIGHT_DECAY', 'THRESHOLD', 'RUN_TEST')}
    configuration.update(parameters=PARAMETERS, architecture='CrackFusionTiny-v1',
        model_source_sha256='458720036d0ad57c89715e8afd4383db4e3a39bcfd9be7e3dab141c034a53f38',
        torch=str(torch.__version__), torchvision=str(torchvision.__version__), device=str(DEVICE),
        manifest_sha256=hashlib.sha256(manifest_bytes).hexdigest(), data_root=str(DATA_ROOT.resolve()))
    identity = hashlib.sha256(json.dumps(configuration, sort_keys=True).encode()).hexdigest()[:16]
    run_dir = OUTPUT_ROOT / ('run_' + identity)
    run_dir.mkdir(parents=True, exist_ok=True)
    save_json(run_dir / 'amp_recovery_fix.json', {'change': 'Skip AMP-overflow updates before gradient clipping; retain scaler recovery.', 'resume': 'Last completed epoch; original run identity retained.'})
    save_json(run_dir / 'configuration.json', configuration)
    (run_dir / 'manifest.csv').write_bytes(manifest_bytes)
    print('Checkpoints:', run_dir.resolve())
    groups = load_manifest(MANIFEST, DATA_ROOT, PROTOCOL, PARENT_ID_PROVENANCE)
    save_json(run_dir / 'manifest_verified.json', {'manifest_sha256': configuration['manifest_sha256'],
        'counts': {split: len(rows) for split, rows in groups.items()}})
    mean, std = training_normalization(groups['train'], DATA_ROOT, run_dir / 'normalization_progress.json')
    configuration.update(mean=mean, std=std)
    save_json(run_dir / 'configuration.json', configuration)
    results = []
    for seed in SEEDS:
        result_path = run_dir / f'metrics_seed_{seed}.json'
        if result_path.exists():
            result = json.loads(result_path.read_text())
            print(f'Reusing completed seed={seed}')
        else:
            try:
                result = train_one_seed(groups, mean, std, seed, run_dir)
            except BaseException as error:
                save_json(run_dir / 'last_error.json', {'seed': seed, 'error': repr(error),
                    'resume': 'Rerun this cell to continue from the last completed epoch.'})
                raise
        results.append(result)
        save_json(run_dir / 'completed_seeds.json', [result['seed'] for result in results])
    summary = {}
    for key in ('macro_f1', 'f1', 'recall', 'balanced_accuracy', 'average_precision'):
        values = [r['metrics'][key] for r in results]
        summary[key] = dict(mean=statistics.mean(values),
                            seed_std=statistics.stdev(values) if len(values) > 1 else None)
    save_json(run_dir / 'summary.json', summary)
    print('Saved:', run_dir.resolve())
    return results, summary

if RUN_TRAINING:
    experiment_results, experiment_summary = run_experiment()
    print(json.dumps(experiment_summary, indent=2))
else:
    print('Real-data training disabled. Set DATA_ROOT, MANIFEST, provenance, and RUN_TRAINING.')


## 8. Optional synthetic training-pipeline check

Disabled by default. Uses temporary files and restores the real training configuration.


In [ ]:
def validate_training_pipeline():
    global DATA_ROOT, MANIFEST, OUTPUT_ROOT, PROTOCOL, PARENT_ID_PROVENANCE
    global BATCH_SIZE, EFFECTIVE_BATCH, MAX_EPOCHS, SEEDS, RUN_TEST
    names = ('DATA_ROOT', 'MANIFEST', 'OUTPUT_ROOT', 'PROTOCOL', 'PARENT_ID_PROVENANCE',
             'BATCH_SIZE', 'EFFECTIVE_BATCH', 'MAX_EPOCHS', 'SEEDS', 'RUN_TEST')
    saved = {name: globals()[name] for name in names}
    try:
        with TemporaryDirectory() as temporary:
            DATA_ROOT = Path(temporary)
            MANIFEST, OUTPUT_ROOT = DATA_ROOT / 'manifest.csv', DATA_ROOT / 'runs'
            PROTOCOL, PARENT_ID_PROVENANCE = 'source_separated', 'Artificial fixture IDs only'
            BATCH_SIZE, EFFECTIVE_BATCH, MAX_EPOCHS, SEEDS, RUN_TEST = 2, 4, 1, (42,), True
            rng = np.random.default_rng(7)
            rows = []
            for split, count in [('train', 6), ('validation', 2), ('test', 2)]:
                for index in range(count):
                    name = f'{split}_{index}.png'
                    Image.fromarray(rng.integers(0, 256, (256, 256, 3), dtype=np.uint8)).save(DATA_ROOT / name)
                    rows.append(dict(path=name, label=index % 2, surface='deck',
                                     parent_id=f'{split}_{index}', split=split))
            def write_rows(items):
                with MANIFEST.open('w', newline='') as stream:
                    writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
                    writer.writeheader(); writer.writerows(items)
            write_rows(rows)
            results, _ = run_experiment()
            assert results[0]['split'] == 'test'
            assert results[0]['metrics']['confusion_matrix']
            checkpoint = next(OUTPUT_ROOT.rglob('best_seed_*.pt'))
            assert torch.load(checkpoint, weights_only=True, map_location='cpu')['parameters'] == PARAMETERS
            predictions = next(OUTPUT_ROOT.rglob('predictions*.csv'))
            with predictions.open() as stream:
                assert len(list(csv.DictReader(stream))) == 2
            for kind in ('parent', 'duplicate'):
                broken = [dict(row) for row in rows]
                if kind == 'parent':
                    broken[-1]['parent_id'] = broken[0]['parent_id']
                else:
                    duplicate = DATA_ROOT / 'duplicate.png'
                    duplicate.write_bytes((DATA_ROOT / rows[0]['path']).read_bytes())
                    broken[-1]['path'] = duplicate.name
                write_rows(broken)
                try:
                    load_manifest(MANIFEST, DATA_ROOT, PROTOCOL, PARENT_ID_PROVENANCE)
                except ValueError as error:
                    assert 'leakage' in str(error).lower() or 'duplicate' in str(error).lower()
                else:
                    raise AssertionError(f'{kind} leakage was accepted')
    finally:
        globals().update(saved)
    return 'passed: train/evaluate/save/predict, parent and exact-duplicate rejection'

if RUN_SOFTWARE_CHECKS:
    validation_report['training_pipeline'] = validate_training_pipeline()
    print(validation_report['training_pipeline'])


## 9. Load trained weights and classify your own image

Use a real-data checkpoint from section 7. A single image prediction is a screening
output, not a crack-width measurement or structural diagnosis. The returned sigmoid
score is not guaranteed calibrated, especially with class-weighted BCE.


In [ ]:
def predict_image(checkpoint_path, image_path):
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    if checkpoint.get('architecture') != 'CrackFusionTiny-v1':
        raise ValueError('Wrong checkpoint architecture')
    net = CrackFusionTiny().to(DEVICE).eval()
    net.load_state_dict(checkpoint['model'])
    _, transform = make_transforms(checkpoint['mean'], checkpoint['std'])
    with Image.open(image_path) as image:
        tensor = transform(image.convert('RGB')).unsqueeze(0).to(DEVICE)
    with torch.inference_mode():
        score = net(tensor).sigmoid().item()
    return dict(crack_score=score, prediction='crack' if score >= checkpoint['threshold'] else 'no crack',
                threshold=checkpoint['threshold'], training_protocol=checkpoint['protocol'])

# predict_image('outputs/crackfusion_tiny/<run>/best_seed_42.pt', 'example.jpg')

# Verify the public inference function with a temporary, explicitly untrained checkpoint.
if RUN_SOFTWARE_CHECKS:
    with TemporaryDirectory() as temporary:
        temporary = Path(temporary)
        inference_net = CrackFusionTiny().eval()
        torch.save(dict(model=inference_net.state_dict(), architecture='CrackFusionTiny-v1',
            mean=[0.5]*3, std=[0.25]*3, threshold=0.5, protocol='synthetic_check'), temporary / 'model.pt')
        Image.new('RGB', (256, 256), color=(140, 140, 140)).save(temporary / 'image.png')
        inference_result = predict_image(temporary / 'model.pt', temporary / 'image.png')
        assert 0 <= inference_result['crack_score'] <= 1
        assert inference_result['training_protocol'] == 'synthetic_check'
        _, inference_transform = make_transforms([0.5]*3, [0.25]*3)
        with Image.open(temporary / 'image.png') as image, torch.inference_mode():
            expected_score = inference_net(inference_transform(image).unsqueeze(0)).sigmoid().item()
        assert abs(inference_result['crack_score'] - expected_score) < 1e-4
    validation_report['custom_image_inference'] = 'passed'
    print('Custom-image preprocessing and checkpoint inference: passed')


## 10. Measured inference latency and validation report

Random weights are sufficient for timing this fixed graph. Timing uses batch 1,
float32, warm-up, and device synchronization; it excludes image preprocessing.
Parameter memory is not peak runtime memory. CUDA peak allocation is measured when
available. CPU/MPS peak memory and MACs/FLOPs are left unreported, not estimated.


In [ ]:
def synchronize():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    elif DEVICE.type == 'mps':
        torch.mps.synchronize()

if RUN_SOFTWARE_CHECKS:
    timing_model = CrackFusionTiny().to(DEVICE).eval()
    timing_input = torch.randn(1, 3, 224, 224, device=DEVICE)
    durations = []
    with torch.inference_mode():
        for _ in range(5):
            timing_model(timing_input)
        synchronize()
        if DEVICE.type == 'cuda':
            torch.cuda.reset_peak_memory_stats()
        for _ in range(20):
            synchronize(); start = time.perf_counter()
            timing_model(timing_input)
            synchronize(); durations.append((time.perf_counter() - start) * 1000)
    validation_report.update(parameters=PARAMETERS, device=str(DEVICE),
        median_batch1_ms=statistics.median(durations), torch=str(torch.__version__),
        cuda_peak_allocated_bytes=torch.cuda.max_memory_allocated() if DEVICE.type == 'cuda' else None,
        research_accuracy_measured=False)
    print(json.dumps(validation_report, indent=2))
    del timing_model


## 11. Results and recovery

Use `best_seed_*.pt` for image inference. `last_seed_*.pt` contains optimizer, scaler, epoch and random states for recovery. A seed is marked complete after its prediction export. Rerun all cells after a runtime reset to reload data and resume.

This uses the audit development split with possible parent-photo overlap. Final research claims require a verified parent mapping and separate evaluation.
